# Lesson 2 — The Bigram Machine

Counts become probabilities, probabilities become dice, and the table starts
writing text that has never existed before.

In [ ]:
# The corpus: Alice in Wonderland (public domain), with a built-in backup.
import urllib.request, re

FALLBACK = ("the small machine counted every letter of the paragraph and then began "
    "to write its own strange sentences about the city and the lake and the long "
    "quiet train ride home it wrote about the coach and the counselor and the "
    "quiet gym at seven in the morning and although every line was gibberish the "
    "shape of the words was english because the counts had come from english ") * 8

try:
    raw = urllib.request.urlopen("https://www.gutenberg.org/files/11/11-0.txt", timeout=15).read().decode("utf-8")
    raw = raw[raw.find("Alice was beginning"):raw.find("THE END")]
    print("Loaded Alice in Wonderland:", len(raw), "characters")
except Exception as e:
    raw = FALLBACK
    print("Download failed (%s) - using the built-in backup corpus." % type(e).__name__)

# Keep only lowercase letters and spaces - 27 symbols total.
corpus = re.sub(r"[^a-z ]+", " ", raw.lower())
corpus = re.sub(r" +", " ", corpus).strip()
print("Cleaned corpus:", len(corpus), "characters")
print(repr(corpus[:100]))

In [ ]:
# Count (lesson 1's move)
counts = {}
for i in range(len(corpus) - 1):
    a, b = corpus[i], corpus[i + 1]
    counts.setdefault(a, {}).setdefault(b, 0)
    counts[a][b] += 1

## Move one: counts to probabilities

Divide each row by its total. Every row becomes betting odds that sum to 1.

In [ ]:
probs = {}
for a, row in counts.items():
    total = sum(row.values())
    probs[a] = {b: n / total for b, n in row.items()}

row_t = sorted(probs["t"].items(), key=lambda kv: -kv[1])[:5]
print("odds after t:", [(b, round(p, 3)) for b, p in row_t])
print("row sums to:", round(sum(probs["t"].values()), 6))

## Move two: roll the dice

Start from a letter, look up its row, pick the next letter weighted by the
odds, repeat. `random.choices` does the weighted pick.

In [ ]:
import random
random.seed()

def generate(start="t", length=200):
    out = start
    cur = start
    for _ in range(length):
        row = probs.get(cur)
        if not row:
            cur = " "
            continue
        letters = list(row.keys())
        weights = list(row.values())
        cur = random.choices(letters, weights=weights)[0]
        out += cur
    return out

print(generate())

## Look closely at the gibberish

Run the cell above five more times. It's wrong the way English is wrong, not
the way static is wrong: word lengths look right, q finds u, vowels show up
on schedule. Everything it knows, it counted.

In [ ]:
# Turn-in helper: generate 20 lines, then pick your three most English-looking
# and the single strangest, and note WHICH COUNTS explain each pick.
for i in range(20):
    print(f"{i+1:2d}.", generate(length=60))